# BEHRT cohort characteristics

Describes the **full modeled cohort** (`patient_sequences_finetune.pkl`) broken down by the four risk classes.

| risk_class | display label |
|---|---|
| 0 | Control |
| 2 | < 1 year |
| 1 | 1–5 years |
| 3 | 5–10 years |

Sections:
1. **Sex distribution** per risk group (sex: 1 = Male, 2 = Female; MarketScan coding).
2. **Age** — age at end of observation (last value of each patient's age sequence).
3. **Comorbidity burden** — number of distinct phenotypes per patient.
4. **Disease prevalence** — % of patients carrying each phenotype, overall and by risk group.

Figures are saved to `publication_figures/`.

In [ ]:
import os, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from collections import Counter

sns.set_theme(style='whitegrid', context='talk')

DATA_DIR   = 'data/behrt'
FIG_DIR    = 'results/figures'
os.makedirs(FIG_DIR, exist_ok=True)

COHORT_PKL = os.path.join(DATA_DIR, 'patient_sequences_finetune.pkl')
PINFO_PKL  = os.path.join(DATA_DIR, 'patient_info_db1.pkl')

SPECIAL_TOKENS = {'PAD', 'CLS', 'SEP', 'MASK', 'UNK'}

# native risk_class -> display label, in the paper's display order
CLASS_LABEL = {0: 'Control', 2: '< 1 year', 1: '1–5 years', 3: '5–10 years'}
CLASS_ORDER = ['Control', '< 1 year', '1–5 years', '5–10 years']
PALETTE     = dict(zip(CLASS_ORDER, sns.color_palette('viridis', n_colors=4)))
SEX_LABEL   = {1: 'Male', 2: 'Female'}

## 1. Load data and reduce to a per-patient table (one pass)

`patient_sequences_finetune.pkl` is ~22 GB; loading it takes a few minutes. In a single pass we build the per-patient scalars (sex, age-at-end, distinct-comorbidity count) **and** accumulate disease-prevalence counters, then drop the heavy frame.

In [ ]:
print('Loading patient_info (sex)...')
with open(PINFO_PKL, 'rb') as f:
    patient_info = pickle.load(f)
sex_dict = {eid: info['sex'] for eid, info in patient_info.items()}
del patient_info

print('Loading cohort sequences (~22 GB, be patient)...')
seq = pd.read_pickle(COHORT_PKL)
print(f'  {len(seq):,} patients | columns: {list(seq.columns)}')

In [ ]:
# Single pass: per-patient scalars + disease prevalence (overall and per risk group)
print('Reducing sequences...')
n = len(seq)
ages = np.empty(n, dtype=np.float32)
ncom = np.empty(n, dtype=np.int32)
overall_counts = Counter()                                # patients carrying each disease
group_counts   = {rc: Counter() for rc in CLASS_LABEL}    # same, split by risk_class
group_n        = {rc: 0 for rc in CLASS_LABEL}            # patients per group

codes_arr = seq['code'].values
ages_arr  = seq['age'].values
rc_arr    = seq['risk_class'].values

for i in range(n):
    uniq = {c for c in codes_arr[i] if c not in SPECIAL_TOKENS}  # distinct diseases for this patient
    ncom[i] = len(uniq)
    a = ages_arr[i]
    ages[i] = int(a[-1]) if len(a) else np.nan
    rc = rc_arr[i]
    overall_counts.update(uniq)
    group_counts[rc].update(uniq)
    group_n[rc] += 1

cohort = pd.DataFrame({
    'enrolid':         seq['enrolid'].values,
    'risk_class':      rc_arr,
    'age_at_end':      ages,
    'n_comorbidities': ncom,
})
cohort['sex']       = cohort['enrolid'].map(sex_dict)
cohort['group']     = cohort['risk_class'].map(CLASS_LABEL)
cohort['sex_label'] = cohort['sex'].map(SEX_LABEL)

del seq, codes_arr, ages_arr
import gc; gc.collect()

print(f"Missing sex: {cohort['sex'].isna().sum():,}")
cohort.to_csv(os.path.join(DATA_DIR, 'cohort_characteristics.csv'), index=False)
cohort.head()

## 2. Table 1 — summary by risk group

In [ ]:
def summarise(g):
    n = len(g)
    female = (g['sex'] == 2).sum()
    return pd.Series({
        'N':              n,
        '% female':       100 * female / g['sex'].notna().sum(),
        'Age median':     g['age_at_end'].median(),
        'Age IQR':        f"{g['age_at_end'].quantile(.25):.0f}–{g['age_at_end'].quantile(.75):.0f}",
        'Comorb. median': g['n_comorbidities'].median(),
        'Comorb. IQR':    f"{g['n_comorbidities'].quantile(.25):.0f}–{g['n_comorbidities'].quantile(.75):.0f}",
    })

table1 = cohort.groupby('group').apply(summarise).reindex(CLASS_ORDER)
table1['N'] = table1['N'].astype(int)
table1.to_csv(os.path.join(DATA_DIR, 'table1_cohort.csv'))
table1.round(1)

## 3. Sex distribution by risk group

In [ ]:
sex_ct  = pd.crosstab(cohort['group'], cohort['sex_label']).reindex(CLASS_ORDER)
sex_pct = sex_ct.div(sex_ct.sum(axis=1), axis=0) * 100
chi2, p, _, _ = stats.chi2_contingency(sex_ct.values)

fig, ax = plt.subplots(figsize=(9, 6))
sex_pct[['Female', 'Male']].plot(kind='bar', stacked=True, ax=ax,
                                  color=['#d8576b', '#3b528b'], width=0.7)
for i, grp in enumerate(CLASS_ORDER):
    ax.text(i, sex_pct.loc[grp, 'Female'] / 2, f"{sex_pct.loc[grp, 'Female']:.0f}%",
            ha='center', va='center', color='white', fontsize=12)
ax.set_ylabel('% of patients'); ax.set_xlabel('')
ax.set_title(f'Sex distribution by risk group\n($\\chi^2$ p = {p:.2e})')
ax.legend(title='Sex', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.xticks(rotation=0); plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'cohort_sex_distribution.png'), dpi=300, bbox_inches='tight')
fig.savefig(os.path.join(FIG_DIR, 'cohort_sex_distribution.pdf'), bbox_inches='tight')
plt.show()
sex_pct.round(1)

## 4. Age (at end of observation) by risk group

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
sns.violinplot(data=cohort, x='group', y='age_at_end', order=CLASS_ORDER,
               palette=PALETTE, cut=0, inner='quartile', ax=ax)
ax.set_ylabel('Age at end of observation (years)'); ax.set_xlabel('')
ax.set_title('Age distribution by risk group')
plt.xticks(rotation=0); plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'cohort_age_distribution.png'), dpi=300, bbox_inches='tight')
fig.savefig(os.path.join(FIG_DIR, 'cohort_age_distribution.pdf'), bbox_inches='tight')
plt.show()

## 5. Comorbidity burden by risk group

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
sns.boxplot(data=cohort, x='group', y='n_comorbidities', order=CLASS_ORDER,
            palette=PALETTE, showfliers=False, ax=ax)
ax.set_ylabel('Distinct comorbidities per patient'); ax.set_xlabel('')
ax.set_title('Comorbidity burden by risk group')
plt.xticks(rotation=0); plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'cohort_comorbidity_burden.png'), dpi=300, bbox_inches='tight')
fig.savefig(os.path.join(FIG_DIR, 'cohort_comorbidity_burden.pdf'), bbox_inches='tight')
plt.show()

## 6. Disease prevalence

**Prevalence of a disease = fraction of patients who carry that phenotype at least once** in their (pre-fracture) history. Built from the `overall_counts` / `group_counts` accumulated during the single pass above.

We produce a prevalence table and three views:
- **6a** Top-N most prevalent diseases overall (horizontal bar).
- **6b** Top-N diseases × risk group prevalence (heatmap).
- **6c** Diseases most *enriched* in the fracture groups vs. controls (prevalence-ratio dot plot).

In [ ]:
TOTAL = len(cohort)

prev = pd.DataFrame({'disease': list(overall_counts.keys()),
                     'n_patients': list(overall_counts.values())})
prev['prevalence_pct'] = 100 * prev['n_patients'] / TOTAL

# per-group prevalence (% within each risk group)
for rc, name in CLASS_LABEL.items():
    gc_ = group_counts[rc]; gn = group_n[rc]
    prev[f'pct_{name}'] = prev['disease'].map(lambda d: 100 * gc_.get(d, 0) / gn)

prev = prev.sort_values('prevalence_pct', ascending=False).reset_index(drop=True)
prev.to_csv(os.path.join(DATA_DIR, 'disease_prevalence.csv'), index=False)
print(f'{len(prev):,} distinct phenotypes')

# ── LaTeX version of the prevalence table ──────────────────────────────────
TOP_N_TEX = 20
tex_cols = ['disease', 'prevalence_pct'] + [f'pct_{name}' for name in CLASS_ORDER]
tex_df = prev.head(TOP_N_TEX)[tex_cols].copy()
tex_df.columns = ['Phenotype', 'Overall'] + CLASS_ORDER

latex_str = tex_df.to_latex(
    index=False,
    float_format='%.1f',
    caption=(f'Prevalence (\\%) of the {TOP_N_TEX} most common phenotypes in the modeled '
             f'cohort (N\\,=\\,{TOTAL:,}), overall and within each risk group.'),
    label='tab:disease_prevalence',
    column_format='l' + 'r' * (len(tex_df.columns) - 1),
    escape=True,
    position='!ht',
)
# booktabs + multi-line header touch-ups
latex_str = latex_str.replace('< 1 year', '$<$1\\,yr').replace('1–5 years', '1--5\\,yr').replace('5–10 years', '5--10\\,yr')

with open(os.path.join(DATA_DIR, 'disease_prevalence_table.tex'), 'w') as f:
    f.write(latex_str)
print('Wrote disease_prevalence_table.tex\n')
print(latex_str)
tex_df.round(1)

In [ ]:
# 6a. Top-N most prevalent diseases overall
TOP_N = 25
top = prev.head(TOP_N).iloc[::-1]

fig, ax = plt.subplots(figsize=(10, 0.45 * TOP_N + 1))
ax.barh(top['disease'], top['prevalence_pct'], color=sns.color_palette('viridis', TOP_N))
for y, v in enumerate(top['prevalence_pct']):
    ax.text(v + 0.3, y, f'{v:.1f}%', va='center', fontsize=11)
ax.set_xlabel('Prevalence (% of cohort)'); ax.set_ylabel('')
ax.set_title(f'Top {TOP_N} most prevalent phenotypes (N = {TOTAL:,})')
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'disease_prevalence_top.png'), dpi=300, bbox_inches='tight')
fig.savefig(os.path.join(FIG_DIR, 'disease_prevalence_top.pdf'), bbox_inches='tight')
plt.show()

In [ ]:
# 6b. Heatmap: top diseases x risk group prevalence
TOP_N_HM = 30
cols = [f'pct_{name}' for name in CLASS_ORDER]
hm = prev.head(TOP_N_HM).set_index('disease')[cols]
hm.columns = CLASS_ORDER

fig, ax = plt.subplots(figsize=(9, 0.45 * TOP_N_HM + 1))
sns.heatmap(hm, annot=True, fmt='.0f', cmap='rocket_r', cbar_kws={'label': 'Prevalence (%)'},
            linewidths=0.4, ax=ax)
ax.set_title(f'Prevalence (%) of top {TOP_N_HM} phenotypes by risk group')
ax.set_xlabel(''); ax.set_ylabel('')
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'disease_prevalence_heatmap.png'), dpi=300, bbox_inches='tight')
fig.savefig(os.path.join(FIG_DIR, 'disease_prevalence_heatmap.pdf'), bbox_inches='tight')
plt.show()

In [ ]:
# 6c. Diseases most enriched in fracture groups vs. controls (prevalence ratio)
MIN_PREV = 1.0   # only consider phenotypes present in >=1% of the cohort (stable ratios)
TOP_ENR  = 20

frac_n = {rc: group_n[rc] for rc in (1, 2, 3)}
tot_frac = sum(frac_n.values())

def frac_pct(d):
    return 100 * sum(group_counts[rc].get(d, 0) for rc in (1, 2, 3)) / tot_frac

enr = prev[prev['prevalence_pct'] >= MIN_PREV].copy()
enr['pct_fracture'] = enr['disease'].map(frac_pct)
enr['ratio'] = enr['pct_fracture'] / enr['pct_Control'].replace(0, np.nan)
enr = enr.dropna(subset=['ratio']).sort_values('ratio', ascending=False)
top_enr = enr.head(TOP_ENR).iloc[::-1]

fig, ax = plt.subplots(figsize=(10, 0.45 * TOP_ENR + 1))
ax.hlines(top_enr['disease'], 1, top_enr['ratio'], color='lightgray', lw=2, zorder=1)
ax.scatter(top_enr['ratio'], top_enr['disease'], color='#b5366c', s=90, zorder=2)
for y, (r, fp, cp) in enumerate(zip(top_enr['ratio'], top_enr['pct_fracture'], top_enr['pct_Control'])):
    ax.text(r + 0.02, y, f'{r:.2f}x  ({fp:.1f}% vs {cp:.1f}%)', va='center', fontsize=10)
ax.axvline(1.0, color='k', ls='--', lw=1)
ax.set_xlabel('Prevalence ratio (fracture groups / control)'); ax.set_ylabel('')
ax.set_title(f'Phenotypes most enriched in fracture patients\n(prevalence ≥ {MIN_PREV:.0f}% of cohort)')
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'disease_enrichment_fracture.png'), dpi=300, bbox_inches='tight')
fig.savefig(os.path.join(FIG_DIR, 'disease_enrichment_fracture.pdf'), bbox_inches='tight')
plt.show()
top_enr[['disease', 'pct_Control', 'pct_fracture', 'ratio']].iloc[::-1].round(2)